In [29]:
import tensorflow_probability as tfp
import tensorflow as tf
import pandas as pd
import numpy as np
import gpflow
from pathlib import Path
import ray
from ABC_GP_IPM.gp_ipm import GP_IPM
from ABC_GP_IPM.abc_gp_ipm import ABC_GP_IPM
from ABC_GP_IPM.calculate_ss import Perted_IPM
from ABC_GP_IPM.gp_cachebasic import predict_y_loaded_cache

In [30]:
df = pd.read_csv('glm_ibm_dataset.csv')

In [31]:
df.describe().round(1)

,z,Repr,Seeds,Surv,z1,age,yr
count,560.0,560.0,16.0,544.0,212.0,560.0,560.0
mean,0.3,0.0,1658.0,0.4,1.3,0.5,2.5
std,1.1,0.2,1812.5,0.5,0.8,0.9,1.6
min,-2.3,0.0,158.0,0.0,-0.9,0.0,1.0
25%,-0.5,0.0,842.2,0.0,0.8,0.0,1.0
50%,0.1,0.0,1039.0,0.0,1.4,0.0,2.0
75%,1.0,0.0,1512.0,1.0,1.9,1.0,4.0
max,3.5,1.0,6257.0,1.0,3.5,4.0,5.0


In [32]:
years = sorted(df['yr'].unique())

In [33]:
POPUdata_dict ={
    i: df.loc[df['yr']==year].copy() for i, year in enumerate(years[:2])
}

In [34]:
prior_scale = tf.constant(1.0, dtype=gpflow.default_float())
prior_loc = tf.constant(0.0, dtype=gpflow.default_float())

NUM_BURNIN = 10
NUM_RESULTS = 5

In [44]:
def XY_m_surv_compu(dataset: pd.DataFrame):
    mask = ((dataset["Repr"] == 0) & dataset["z"].notna() & dataset["Surv"].notna())
    X = dataset.loc[mask, ["z"]].to_numpy(dtype=np.float64)
    Y = dataset.loc[mask, ["Surv"]].to_numpy(dtype=np.float64)
    return X, Y

def build_new_m_surv(dataset: pd.DataFrame):
    mean_function = gpflow.mean_functions.Constant(c=0.0)
    rbf = gpflow.kernels.SquaredExponential()
    X, Y = XY_m_surv_compu(dataset)
    model = gpflow.models.GPMC(data=(X, Y), 
                              kernel=rbf, 
                              mean_function=mean_function, 
                              likelihood=gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
    model.kernel.lengthscales.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.kernel.variance.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.mean_function.c.prior = (tfp.distributions.Normal(prior_loc, prior_scale))
    return model

In [45]:
def XY_m_repr_compu(dataset: pd.DataFrame):
    mask = (dataset["z"].notna() & dataset["Repr"].notna())
    X = dataset.loc[mask, ["z"]].to_numpy(dtype=np.float64)
    Y = dataset.loc[mask, ["Repr"]].to_numpy(dtype=np.float64)
    return X, Y

def build_new_m_repr(dataset: pd.DataFrame):
    mean_function = gpflow.mean_functions.Constant(c=0.0)
    rbf = gpflow.kernels.SquaredExponential()
    X, Y = XY_m_repr_compu(dataset)
    model = gpflow.models.GPMC(data=(X, Y), 
                              kernel=rbf, 
                              mean_function=mean_function, 
                              likelihood=gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
    model.kernel.lengthscales.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.kernel.variance.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.mean_function.c.prior = (tfp.distributions.Normal(prior_loc, prior_scale))
    return model

In [46]:
def XY_m_seed_compu(dataset: pd.DataFrame):
    mask = ((dataset["Repr"] == 1) & dataset["z"].notna() & dataset["Seeds"].notna())
    X = dataset.loc[mask, ["z"]].to_numpy(dtype=np.float64)
    Y = dataset.loc[mask, ["Seeds"]].to_numpy(dtype=np.float64)
    return X, Y

def build_new_m_seed(dataset: pd.DataFrame):
    mean_function = gpflow.mean_functions.Constant(c=0.0)
    rbf = gpflow.kernels.SquaredExponential()
    X, Y = XY_m_seed_compu(dataset)
    model = gpflow.models.GPMC(data=(X, Y), 
                              kernel=rbf, 
                              mean_function=mean_function, 
                              likelihood=gpflow.likelihoods.Poisson())
    model.kernel.lengthscales.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.kernel.variance.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.mean_function.c.prior = (tfp.distributions.Normal(prior_loc, prior_scale))
    return model

In [47]:
def XY_m_grw_compu(dataset: pd.DataFrame):
    mask = ((dataset["Surv"] == 1) & dataset["z"].notna() & dataset["z1"].notna())
    X = dataset.loc[mask, ["z"]].to_numpy(dtype=np.float64)
    Y = dataset.loc[mask, ["z1"]].to_numpy(dtype=np.float64)
    return X, Y

def build_new_m_grw(dataset: pd.DataFrame):
    mean_function = gpflow.mean_functions.Constant(c=0.0)
    rbf = gpflow.kernels.SquaredExponential()
    X, Y = XY_m_grw_compu(dataset)
    model = gpflow.models.GPR(data=(X, Y), kernel=rbf, mean_function=mean_function)
    model.kernel.lengthscales.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.kernel.variance.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.likelihood.variance.prior = (tfp.distributions.HalfNormal(prior_scale))
    model.mean_function.c.prior = (tfp.distributions.Normal(prior_loc, prior_scale))
    return model

In [48]:
X_growth, Y_growth = XY_m_grw_compu(df)
X_surv, Y_surv = XY_m_surv_compu(df)
X_repr, Y_repr = XY_m_repr_compu(df)
X_seed, Y_seed = XY_m_seed_compu(df)

In [49]:
growth_mle = gpflow.models.GPR(data=(X_growth, Y_growth), kernel=rbf, mean_function=mean_function)
optimizer = gpflow.optimizers.Scipy()
optimizer.minimize(growth_mle.training_loss, growth_mle.trainable_variables)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 225.81482826828497
        x: [ 1.397e+00 -4.773e-01 -5.663e-01  9.935e-01]
      nit: 11
      jac: [ 8.342e-05  4.503e-05 -1.027e-03  5.941e-05]
     nfev: 14
     njev: 14
 hess_inv: <4x4 LbfgsInvHessProduct with dtype=float64>

In [51]:
mean_function = gpflow.mean_functions.Constant(c=0.0)
rbf = gpflow.kernels.SquaredExponential()
surv_mle = gpflow.models.VGP(data=(X_surv, Y_surv), kernel=rbf, mean_function=mean_function, likelihood=gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
optimizer = gpflow.optimizers.Scipy()
optimizer.minimize(surv_mle.training_loss, surv_mle.trainable_variables)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 336.01936577337295
        x: [-1.226e+00  4.526e-01 ...  1.619e-01 -2.844e-01]
      nit: 55
      jac: [ 6.799e-05  1.042e-03 ...  4.887e-04  9.622e-04]
     nfev: 60
     njev: 60
 hess_inv: <148787x148787 LbfgsInvHessProduct with dtype=float64>

In [54]:
mean_function = gpflow.mean_functions.Constant(c=0.0)
rbf = gpflow.kernels.SquaredExponential()
repr_mle = gpflow.models.VGP(data=(X_repr, Y_repr), kernel=rbf, mean_function=mean_function, likelihood=gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
optimizer = gpflow.optimizers.Scipy()
optimizer.minimize(repr_mle.training_loss, repr_mle.trainable_variables)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 23.14069392719837
        x: [-1.004e+00  3.156e-01 ...  1.526e+02 -4.113e+00]
      nit: 733
      jac: [-3.016e-04 -4.179e-04 ... -8.849e-05  4.291e-04]
     nfev: 793
     njev: 793
 hess_inv: <157643x157643 LbfgsInvHessProduct with dtype=float64>

In [55]:
mean_function = gpflow.mean_functions.Constant(c=0.0)
rbf = gpflow.kernels.SquaredExponential()
seed_mle = gpflow.models.VGP(data=(X_seed, Y_seed), kernel=rbf, mean_function=mean_function, likelihood=gpflow.likelihoods.Poisson())
optimizer = gpflow.optimizers.Scipy()
optimizer.minimize(seed_mle.training_loss, seed_mle.trainable_variables)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 96.97041038873209
        x: [-7.803e-02 -1.907e+00 ...  2.489e+01  7.367e+00]
      nit: 4485
      jac: [ 1.024e-01  3.809e-02 ... -1.467e-02  4.508e-02]
     nfev: 4987
     njev: 4987
 hess_inv: <155x155 LbfgsInvHessProduct with dtype=float64>

In [56]:
growth_bayesian = build_new_m_grw(df)
gpflow.utilities.print_summary(growth_bayesian)

╒═════════════════════════╤═══════════╤══════════════════╤════════════╤═════════════╤═════════╤═════════╤═════════╕
│ name                    │ class     │ transform        │ prior      │ trainable   │ shape   │ dtype   │   value │
╞═════════════════════════╪═══════════╪══════════════════╪════════════╪═════════════╪═════════╪═════════╪═════════╡
│ GPR.mean_function.c     │ Parameter │ Identity         │ Normal     │ True        │ ()      │ float64 │       0 │
├─────────────────────────┼───────────┼──────────────────┼────────────┼─────────────┼─────────┼─────────┼─────────┤
│ GPR.kernel.variance     │ Parameter │ Softplus         │ HalfNormal │ True        │ ()      │ float64 │       1 │
├─────────────────────────┼───────────┼──────────────────┼────────────┼─────────────┼─────────┼─────────┼─────────┤
│ GPR.kernel.lengthscales │ Parameter │ Softplus         │ HalfNormal │ True        │ ()      │ float64 │       1 │
├─────────────────────────┼───────────┼──────────────────┼────────────┼─

In [57]:
surv_bayesian = build_new_m_surv(df)
gpflow.utilities.print_summary(surv_bayesian)

╒══════════════════════════╤═══════════╤═════════════╤════════════╤═════════════╤══════════╤═════════╤═════════╕
│ name                     │ class     │ transform   │ prior      │ trainable   │ shape    │ dtype   │ value   │
╞══════════════════════════╪═══════════╪═════════════╪════════════╪═════════════╪══════════╪═════════╪═════════╡
│ GPMC.mean_function.c     │ Parameter │ Identity    │ Normal     │ True        │ ()       │ float64 │ 0.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼──────────┼─────────┼─────────┤
│ GPMC.kernel.variance     │ Parameter │ Softplus    │ HalfNormal │ True        │ ()       │ float64 │ 1.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼──────────┼─────────┼─────────┤
│ GPMC.kernel.lengthscales │ Parameter │ Softplus    │ HalfNormal │ True        │ ()       │ float64 │ 1.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼──────────┼────

In [58]:
repr_bayesian = build_new_m_repr(df)
gpflow.utilities.print_summary(repr_bayesian)

╒══════════════════════════╤═══════════╤═════════════╤════════════╤═════════════╤══════════╤═════════╤═════════╕
│ name                     │ class     │ transform   │ prior      │ trainable   │ shape    │ dtype   │ value   │
╞══════════════════════════╪═══════════╪═════════════╪════════════╪═════════════╪══════════╪═════════╪═════════╡
│ GPMC.mean_function.c     │ Parameter │ Identity    │ Normal     │ True        │ ()       │ float64 │ 0.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼──────────┼─────────┼─────────┤
│ GPMC.kernel.variance     │ Parameter │ Softplus    │ HalfNormal │ True        │ ()       │ float64 │ 1.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼──────────┼─────────┼─────────┤
│ GPMC.kernel.lengthscales │ Parameter │ Softplus    │ HalfNormal │ True        │ ()       │ float64 │ 1.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼──────────┼────

In [59]:
seed_bayesian = build_new_m_seed(df)
gpflow.utilities.print_summary(seed_bayesian)

╒══════════════════════════╤═══════════╤═════════════╤════════════╤═════════════╤═════════╤═════════╤═════════╕
│ name                     │ class     │ transform   │ prior      │ trainable   │ shape   │ dtype   │ value   │
╞══════════════════════════╪═══════════╪═════════════╪════════════╪═════════════╪═════════╪═════════╪═════════╡
│ GPMC.mean_function.c     │ Parameter │ Identity    │ Normal     │ True        │ ()      │ float64 │ 0.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼─────────┼─────────┼─────────┤
│ GPMC.kernel.variance     │ Parameter │ Softplus    │ HalfNormal │ True        │ ()      │ float64 │ 1.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼─────────┼─────────┼─────────┤
│ GPMC.kernel.lengthscales │ Parameter │ Softplus    │ HalfNormal │ True        │ ()      │ float64 │ 1.0     │
├──────────────────────────┼───────────┼─────────────┼────────────┼─────────────┼─────────┼─────────┼───

In [60]:
def sample(model):
    helper = gpflow.optimizers.SamplingHelper(model.log_posterior_density, 
                                              model.trainable_parameters)
    hmc_kernel = tfp.mcmc.HamiltonianMonteCarlo(target_log_prob_fn=helper.target_log_prob_fn,
                                                num_leapfrog_steps = 5,
                                                step_size=0.01,)
    adaptive_hmc = tfp.mcmc.SimpleStepSizeAdaptation(inner_kernel=hmc_kernel, 
                                                    num_adaptation_steps=int(0.8 * NUM_BURNIN), 
                                                    target_accept_prob=tf.constant(0.8, dtype=gpflow.default_float()))
    @tf.function
    def run_growth_nuts():
        return tfp.mcmc.sample_chain(num_results=NUM_RESULTS,
                                    num_burnin_steps=NUM_BURNIN,
                                    current_state=helper.current_state,
                                    kernel=adaptive_hmc,
                                    trace_fn=lambda _, kernel_results: kernel_results.inner_results.is_accepted,
                                    seed=123)
    samples, accepted = run_growth_nuts()

    print("Acceptance rate:", float(tf.reduce_mean(tf.cast(accepted, tf.float64))))
    print("Raw sample shapes:", [sample.shape for sample in samples])
    return samples

In [61]:
growth_samples = sample(growth_bayesian)
surv_samples = sample(surv_bayesian)
repr_samples = sample(repr_bayesian)
seed_samples = sample(seed_bayesian)

Acceptance rate: 1.0
Raw sample shapes: [TensorShape([5]), TensorShape([5]), TensorShape([5]), TensorShape([5])]
Acceptance rate: 1.0
Raw sample shapes: [TensorShape([5, 544, 1]), TensorShape([5]), TensorShape([5]), TensorShape([5])]
Acceptance rate: 1.0
Raw sample shapes: [TensorShape([5, 560, 1]), TensorShape([5]), TensorShape([5]), TensorShape([5])]
Acceptance rate: 0.0
Raw sample shapes: [TensorShape([5, 16, 1]), TensorShape([5]), TensorShape([5]), TensorShape([5])]


In [62]:
MCMC_samples = {
    'm_grw': growth_samples,
    'm_surv': surv_samples,
    'm_repr': repr_samples,
    'm_seed': seed_samples,
    }

In [63]:
GPmodel_mle = {
    'm_grw': growth_mle,
    'm_surv': surv_mle,
    'm_repr': repr_mle,
    'm_seed': seed_mle,
    }
GPmodel = {
    'm_grw': growth_bayesian,
    'm_surv': surv_bayesian,
    'm_repr': repr_bayesian,
    'm_seed': seed_bayesian,
    }

full_gp_ipm = GP_IPM(popu_data=df, 
                       POPUdata_dict=POPUdata_dict, 
                       GPmodel_mle=GPmodel_mle, 
                       GPmodel=GPmodel, 
                       MCMC_samples=MCMC_samples)

full_gp_ipm.add_XY_compu(XY_m_grw_compu)
full_gp_ipm.add_XY_compu(XY_m_surv_compu)
full_gp_ipm.add_XY_compu(XY_m_repr_compu)
full_gp_ipm.add_XY_compu(XY_m_seed_compu)

full_gp_ipm.add_model_build(build_new_m_grw)
full_gp_ipm.add_model_build(build_new_m_surv)
full_gp_ipm.add_model_build(build_new_m_repr)
full_gp_ipm.add_model_build(build_new_m_seed)

print(full_gp_ipm.extract_names)

['m_grw', 'm_surv', 'm_repr', 'm_seed']


In [64]:
recruit_data = df.loc[df["age"] == 0, "z"].dropna()
recruit_mean = float(recruit_data.mean())
recruit_sd = float(recruit_data.std())
recruit_probability = float((df["age"] == 0).sum() / df["Seeds"].sum())

print(recruit_mean, recruit_sd, recruit_probability)

-0.20405557934587937 0.7501400661915896 0.014776839565741858


In [65]:
def model_predict_y(model, X, cached):
    if cached:
        mean, variance = predict_y_loaded_cache(model=model, Xnew=X, Cache=model.cache)
    else:
        mean, variance = model.predict_y(X)
    return mean.numpy().reshape(-1), variance.numpy().reshape(-1)

In [66]:
def simulate_one_step(dataset, models, cached):
    z = dataset["z"].to_numpy(dtype=np.float64)
    age = dataset["age"].to_numpy(dtype=int) if "age" in dataset else np.zeros(len(dataset), dtype=int)
    X = z.reshape(-1, 1)

    reproduction_probability, _ = model_predict_y(models["m_repr"], X, cached)
    reproduction_probability = np.clip(reproduction_probability, 0.0, 1.0)
    reproduced = np.random.binomial(n=1, p=reproduction_probability)

    seeds = np.full(len(z), np.nan)
    reproduction_mask = (reproduced == 1)
    if np.any(reproduction_mask):
        expected_seeds, _ = model_predict_y(models["m_seed"], X[reproduction_mask], cached)
        expected_seeds = np.clip(expected_seeds, 0.0, 100000.0)
        seeds[reproduction_mask] = np.random.poisson(lam=expected_seeds)

    survived = np.full(len(z), np.nan)
    nonreproduction_mask = (reproduced == 0)
    if np.any(nonreproduction_mask):
        survival_probability, _ = model_predict_y(models["m_surv"], X[nonreproduction_mask], cached)
        survival_probability = np.clip(survival_probability, 0.0, 1.0)
        survived[nonreproduction_mask] = np.random.binomial(n=1, p=survival_probability)

    alive = ((reproduced == 0) & (survived == 1))
    z1 = np.full(len(z), np.nan)
    if np.any(alive):
        growth_mean, growth_variance = model_predict_y(models["m_grw"], X[alive], cached)
        growth_variance = np.maximum(growth_variance, 0.0)
        z1[alive] = np.random.normal(loc=growth_mean, scale=np.sqrt(growth_variance))

    return pd.DataFrame({"z": z, "Repr": reproduced, "Seeds": seeds, "Surv": survived, "z1": z1, "age": age, "alive": alive})

In [67]:
def IBM_1step_cache(dataset, models):
    return simulate_one_step(dataset, models, cached=True)

def IBM_1step_uncached(dataset, models):
    return simulate_one_step(dataset, models, cached=False)

In [68]:
def population_structure(data_simu, time_step):
    survivors = data_simu.loc[data_simu["alive"], ["z1", "age"]].copy()
    survivor_z = survivors["z1"].to_numpy(dtype=np.float64)
    survivor_age = survivors["age"].to_numpy(dtype=int) + 1

    total_seeds = min(int(np.nansum(data_simu["Seeds"])), 1000000)
    number_recruits = np.random.binomial(n=total_seeds, p=recruit_probability)
    recruit_z = np.random.normal(loc=recruit_mean, scale=recruit_sd, size=number_recruits)
    recruit_age = np.zeros(number_recruits, dtype=int)

    next_z = np.concatenate([survivor_z, recruit_z])
    next_age = np.concatenate([survivor_age, recruit_age])
    return pd.DataFrame({"z": next_z, "age": next_age})

In [69]:
def distribution_statistics(values):
    values = pd.Series(values).dropna().to_numpy(dtype=np.float64)
    if len(values) < 2:
        return np.array([1e6, 1e6, 1e6])
    return np.array([np.mean(values), np.std(values), np.median(values)])

def abc_summary_statistics(exp_dataset, simu_dataset):
    observed = distribution_statistics(exp_dataset["z1"])
    simulated = distribution_statistics(simu_dataset["z1"])
    return np.abs(observed - simulated)

In [70]:
abc_summary_names = ["mean_z1", "sd_z1", "median_z1"]
initial_population = df.loc[df["yr"] == years[0], ["z", "age"]].copy()

In [71]:
run_directory = Path("/tmp/abc_gp_demo")
cache_path = run_directory / "cache"
details_path = run_directory / "details"

full_gp_ipm.calculate_cache(file_path=str(cache_path))

Caching cholesky decomposition...
Done


In [72]:
abc_model = ABC_GP_IPM(GP_IPM_instance=full_gp_ipm, 
                       fun_ss=abc_summary_statistics, 
                       name_ss=abc_summary_names, 
                       ini_popu=initial_population, 
                       fun_IBM_cache=IBM_1step_cache, 
                       fun_structure=population_structure, 
                       cache_path=str(cache_path))

In [73]:
ray.init(num_cpus=1, include_dashboard=False, ignore_reinit_error=True)

abc_indices, abc_threshold = abc_model.ABC_PMC(quantiles=np.array([1.0]), 
                                               n_particles=np.array([4]), 
                                               num_cores=1, 
                                               details=False, 
                                               smallest_unit=1, 
                                               file_path2store=str(details_path))

abc_weights = np.asarray(abc_model.weight)

ray.shutdown()

2026-07-28 22:59:49,130	INFO worker.py:2024 -- Started a local Ray instance.


c=0 22:59:49


(raylet) [2026-07-28 22:59:57,361 E 9317 3914815] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-28_22-59-42_673503_2212 is over 95% full, available space: 5.36514 GB; capacity: 228.274 GB. Object creation will fail if spilling is required.
(para_ipmmcmc_whole pid=9339) /opt/anaconda3/envs/abcgp/lib/python3.11/site-packages/gpflow/versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
(para_ipmmcmc_whole pid=9339)   import pkg_resources


Total: 4
Left: 4
Unique: 4 



In [74]:
print("ABC indices shape:", abc_indices.shape)
print("ABC indices:")
print(abc_indices)
print("Final threshold:", abc_threshold)
print("Weights:", abc_weights)
print("Weights sum:", abc_weights.sum())

ABC indices shape: (4, 4)
ABC indices:
[[0 1 3 2]
 [1 3 0 4]
 [2 1 1 0]
 [4 0 4 3]]
Final threshold: 26.686292857078413
Weights: [0.25 0.25 0.25 0.25]
Weights sum: 1.0
